In [15]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv

In [16]:
# load.env-file
load_dotenv()

schema = "sql_cities"
host = "127.0.0.1"
user = "root"
password = os.getenv("DB_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [17]:
german_cities = ['Berlin', 'Hamburg', 'Munich']
cities_json = []     # Empty list, on each iteration we will add the current API response to it

for german_city in german_cities:
  url = "https://api.api-ninjas.com/v1/city"
  header={"X-Api-Key": "x-api-key"}
  querystring = {"country" : "DE", "name" : german_city}   # Passing the current iteration city name as the value in the 'querystring' dictionary
  city_request = requests.request("GET", url, headers=header, params=querystring)
  city_json = city_request.json()
  cities_json += city_json    # Adding current iteration response to list of responses from all cities together, using '+' to add lists to one another

cities_json

[{'name': 'Berlin',
  'latitude': 52.5167,
  'longitude': 13.3833,
  'country': 'DE',
  'population': 3644826,
  'region': 'N/A',
  'is_capital': True},
 {'name': 'Hamburg',
  'latitude': 53.55,
  'longitude': 10.0,
  'country': 'DE',
  'population': 1841179,
  'region': 'N/A',
  'is_capital': False},
 {'name': 'Munich',
  'latitude': 48.1372,
  'longitude': 11.5755,
  'country': 'DE',
  'population': 1471508,
  'region': 'Bavaria',
  'is_capital': False}]

In [18]:
cities_general_df = pd.DataFrame(cities_json)
cities_general_df

,name,latitude,longitude,country,population,region,is_capital
0,Berlin,52.5167,13.3833,DE,3644826,N/A,True
1,Hamburg,53.5500,10.0000,DE,1841179,N/A,False
2,Munich,48.1372,11.5755,DE,1471508,Bavaria,False


In [22]:
cities_general_df.to_sql('cities_general',
                  if_exists='append',
                  con=connection_string,
                  index=False)

3

In [20]:
cities = {"Berlin": "DE", "Hamburg": "DE", "Munich": "DE"}

def city_weather_request(cities):
    cities_json = []

# API-Keys
    CITY_API_KEY = "city_api_key"
    WEATHER_API_KEY = "weather_api_key"

# API endpoints
    city_url = "https://api.api-ninjas.com/v1/city"
    weather_url = "https://api.openweathermap.org/data/2.5/forecast"

# Header for Ninjas API
    headers = {"X-Api-Key": CITY_API_KEY}

    for city_name, country in cities.items():

    # 1. Request city details
        city_params = {"country": country, "name": city_name}

        city_response = requests.get(city_url, headers=headers, params=city_params)
        city_response.raise_for_status()
        city_data = city_response.json()

        if not city_data:
            print(f"City {city_name} not found")
            continue

        city = city_data[0]

        lat = city["latitude"]
        lon = city["longitude"]

    # 2. Request weather forecast
        weather_params = {"lat": lat, "lon": lon, "appid": WEATHER_API_KEY, "units": "metric"}

        weather_response = requests.get(weather_url, params=weather_params)
        weather_response.raise_for_status()
        weather_data = weather_response.json()

    # 3. Combine both responses
        cities_json.append({"city_details": city, "weather": weather_data})

# 4. Create output table
    rows = []

    for item in cities_json:
        city = item["city_details"]
        weather = item["weather"]

        for forecast in weather["list"]:
            rows.append({
                "City": city["name"],
                "Country": city.get("country"),
                "DTG": forecast["dt_txt"],
                "Temp_C": forecast["main"]["temp"],
                "Feels_Like_C": forecast["main"]["feels_like"],
                "Humid_pct": forecast["main"]["humidity"],
                "Outlook": forecast["weather"][0]["description"],
                "Wind_Speed_ms": forecast["wind"]["speed"]})
    return pd.DataFrame(rows)

cities_all_weather_data_df = city_weather_request(cities)
display(cities_all_weather_data_df)

,City,Country,DTG,Temp_C,Feels_Like_C,Humid_pct,Outlook,Wind_Speed_ms
0,Berlin,DE,2026-10-21 15:00:00,13.88,13.52,84,overcast clouds,1.09
1,Berlin,DE,2026-10-21 18:00:00,13.16,12.78,86,overcast clouds,1.42
2,Berlin,DE,2026-10-21 21:00:00,12.03,11.69,92,light rain,2.59
3,Berlin,DE,2026-10-22 00:00:00,11.10,10.75,95,light rain,3.95
4,Berlin,DE,2026-10-22 03:00:00,11.04,10.71,96,light rain,4.07
...,...,...,...,...,...,...,...,...
115,Munich,DE,2026-10-26 00:00:00,6.34,3.87,95,clear sky,3.37
116,Munich,DE,2026-10-26 03:00:00,5.14,2.90,95,clear sky,2.70
117,Munich,DE,2026-10-26 06:00:00,3.62,2.21,98,broken clouds,1.62
118,Munich,DE,2026-10-26 09:00:00,9.29,9.29,84,overcast clouds,0.94


In [23]:
cities_all_weather_data_df.to_sql('cities_all_weather_data',
                  if_exists='append',
                  con=connection_string,
                  index=False)

120